# 03 — Generate Figure 1: SLC-40 and BCHH deployment geometry

This notebook generates Figure 1 from the standardized geometry products
written by Notebook 02. It uses the existing `figure1_utils` plotting
functions while explicitly adapting the normalized metadata schema to the
columns required by those functions.

**Dependencies:** run Notebooks 01 and 02 first.

07_generate_figure1_cleaned.ipynb
Useful but inconsistently named and numbered.
The notebook title says:
# 03 — Generate Figure 1
while the filename begins with 07_.
That is a clear sign of workflow drift.
Two reasonable options:
S03_generate_deployment_geometry_figure.ipynb
or merge the code into Notebook 12.
I lean toward keeping it separate because map/geometry figures often have different dependencies and are tedious to regenerate. Rename it:
12a_generate_deployment_geometry_figure.ipynb
if it remains part of the paper production set.
Verdict: keep or merge; fix title immediately.

## 1. Imports and project paths

The notebook may be run from either the project root or the `notebooks/`
directory.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from pyproj import CRS, Geod, Transformer

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from project_config import ensure_output_dirs
from geometry_products import read_geometry_products
from figure1_utils import (
    get_sensor_location,
    make_figure1,
    save_figure,
)

PATHS = ensure_output_dirs(PROJECT_ROOT)
DERIVED_DIR = PATHS["derived"]
FIGURE_DIR = PATHS["figures"]

WGS84 = CRS.from_epsg(4326)
UTM17N = CRS.from_epsg(32617)

geod = Geod(ellps="WGS84")
ll_to_utm = Transformer.from_crs(
    WGS84,
    UTM17N,
    always_xy=True,
)
utm_to_ll = Transformer.from_crs(
    UTM17N,
    WGS84,
    always_xy=True,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Derived data: {DERIVED_DIR}")
print(f"Figure output: {FIGURE_DIR}")

## 2. Load the standardized geometry products

Notebook 02 writes the event inventory and normalized channel, station, and
KML-location tables. Figure 1 consumes those products directly rather than
rereading StationXML or KML independently.

In [ ]:
(
    inventory_event,
    channels_df,
    stations_df,
    locations_df,
) = read_geometry_products(DERIVED_DIR)

print("Channel columns:")
print(channels_df.columns.tolist())

print("\nLocation columns:")
print(locations_df.columns.tolist())

display(stations_df)
display(channels_df)
display(locations_df)

## 3. Adapt the channel table for Figure 1

The standardized channel table uses descriptive column names such as
`latitude` and `longitude`. The existing Figure 1 utility expects the
concise plotting schema `sensor`, `label`, `lat`, `lon`, `easting`, and
`northing`. This cell creates those derived fields explicitly.

The three seismic components represent one physical seismometer location,
so they are collapsed to a single `Seismometer` row.

In [ ]:
CHANNEL_TO_SENSOR = {
    "DHZ": "Seismometer",
    "DHN": "Seismometer",
    "DHE": "Seismometer",
    "HHZ": "Seismometer",
    "HHN": "Seismometer",
    "HHE": "Seismometer",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
    "HD1": "HD1",
    "HD2": "HD2",
    "HD3": "HD3",
}

CHANNEL_TO_LABEL = {
    "DHZ": "BCHH",
    "DHN": "BCHH",
    "DHE": "BCHH",
    "HHZ": "BCHH",
    "HHN": "BCHH",
    "HHE": "BCHH",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
    "HD1": "HD1",
    "HD2": "HD2",
    "HD3": "HD3",
}

required_channel_columns = {
    "channel",
    "latitude",
    "longitude",
}
missing = required_channel_columns.difference(channels_df.columns)
if missing:
    raise KeyError(
        "The standardized channel table is missing required columns: "
        f"{sorted(missing)}"
    )

channels_for_map = channels_df.copy()
channel_codes = (
    channels_for_map["channel"]
    .astype(str)
    .str.upper()
)

channels_for_map["sensor"] = channel_codes.map(CHANNEL_TO_SENSOR)
channels_for_map["label"] = channel_codes.map(CHANNEL_TO_LABEL)
channels_for_map["lat"] = channels_for_map["latitude"].astype(float)
channels_for_map["lon"] = channels_for_map["longitude"].astype(float)

eastings, northings = ll_to_utm.transform(
    channels_for_map["lon"].to_numpy(),
    channels_for_map["lat"].to_numpy(),
)
channels_for_map["easting"] = eastings
channels_for_map["northing"] = northings

unmapped = channels_for_map.loc[
    channels_for_map["sensor"].isna(),
    ["seed_id", "channel", "sensor_description"],
]
if not unmapped.empty:
    print("Ignoring channels that are not used in Figure 1:")
    display(unmapped)

mapped = channels_for_map.dropna(subset=["sensor"]).copy()

seismometer_rows = mapped.loc[
    mapped["sensor"] == "Seismometer"
]
if seismometer_rows.empty:
    raise ValueError("No seismic component was mapped to 'Seismometer'.")

# All three components are co-located; retain one representative row.
seismometer_row = seismometer_rows.iloc[[0]].copy()

infrasound_rows = (
    mapped.loc[mapped["sensor"].isin(["HD1", "HD2", "HD3"])]
    .sort_values("sensor")
    .drop_duplicates(subset=["sensor"])
)

bchh_sensors_df = pd.concat(
    [seismometer_row, infrasound_rows],
    ignore_index=True,
)

expected_sensors = {"Seismometer", "HD1", "HD2", "HD3"}
actual_sensors = set(bchh_sensors_df["sensor"])
if actual_sensors != expected_sensors:
    raise ValueError(
        "Expected exactly these physical sensors: "
        f"{sorted(expected_sensors)}; found {sorted(actual_sensors)}"
    )

figure1_columns = [
    "sensor",
    "label",
    "lat",
    "lon",
    "easting",
    "northing",
    "channel",
    "seed_id",
    "sensor_description",
]
display(bchh_sensors_df[figure1_columns])

## 4. Extract the launch-pad and BCHH reference locations

In [ ]:
def get_unique_kml_location(
    dataframe: pd.DataFrame,
    kml_id: str,
) -> dict:
    matches = dataframe.loc[
        dataframe["kml_id"]
        .astype(str)
        .str.upper()
        .eq(kml_id.upper())
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one KML placemark {kml_id!r}; "
            f"found {len(matches)}."
        )

    row = matches.iloc[0].copy()

    # Normalize possible coordinate-column conventions.
    lat_key = (
        "lat"
        if "lat" in row.index
        else "latitude"
    )
    lon_key = (
        "lon"
        if "lon" in row.index
        else "longitude"
    )

    return {
        **row.to_dict(),
        "name": str(row.get("name", kml_id)),
        "lat": float(row[lat_key]),
        "lon": float(row[lon_key]),
    }


SLC40 = get_unique_kml_location(locations_df, "SLC40")
SLC41 = get_unique_kml_location(locations_df, "SLC41")

BCHH = get_sensor_location(
    bchh_sensors_df,
    sensor_name="Seismometer",
    display_name="BCHH",
)

print("SLC-40:", SLC40)
print("SLC-41:", SLC41)
print("BCHH:", BCHH)

## 5. Generate and save Figure 1

Geometry is fixed by the products above. Presentation changes should be made
in `figure1_utils.make_figure1()` or through its supported arguments, not by
redefining coordinates in this notebook.

In [ ]:
fig, axes = make_figure1(
    slc40=SLC40,
    slc41=SLC41,
    bchh=BCHH,
    bchh_sensors=bchh_sensors_df,
    geod=geod,
    ll_to_utm=ll_to_utm,
    utm_to_ll=utm_to_ll,
)

output_paths = save_figure(
    fig=fig,
    output_directory=FIGURE_DIR,
    filename_stem="fig01_slc40_bchh_location",
    extensions=("png", "pdf"),
    dpi=300,
)

print("Saved Figure 1:")
for path in output_paths:
    print(f"  {Path(path).resolve()}")

plt.show()

## Outputs

- `outputs/figures/fig01_slc40_bchh_location.png`
- `outputs/figures/fig01_slc40_bchh_location.pdf`

If only geometry metadata changes, rerun Notebook 02 and this notebook.
Instrument correction and downstream waveform measurements do not need to be
repeated.